# Chapter 4 — Financial Distress and Altman Z-Score

Today’s workflow is familiar: **load the 50 firms → keep each firm’s latest fiscal period → map the accounting inputs → calculate raw Altman Z → add sector context**.

The raw Altman Z is an **absolute distress screen**. The industry-adjusted measure is a **relative diagnostic**. Neither score proves bankruptcy, explains the cause of weakness, or replaces further analysis.

This notebook does **not** calculate the Campbell model, STA, SNOA, PMAN, or a combined Chapter 3/4 rank.


## 1. Load the same Yahoo Finance classroom data used in Chapter 3

We reuse the same 50-firm universe, public raw GitHub URL, pandas import, and full-history copy. This makes the beginning of Chapter 4 familiar.


In [ ]:
import numpy as np
import pandas as pd

EXPECTED_UNIVERSE_COUNT = 50
MIN_SECTOR_OBSERVATIONS = 3
DISTRESS_BOUNDARY = 1.81
ORIGINAL_CUTOFF = 2.675
DATA_URL = "https://raw.githubusercontent.com/x2003xuy/acct3343-public/main/data/acct3343_financial_data_50.csv"
VALIDATION_SNAPSHOT_SHA256 = "b48ba2400340a2447e781abf82333e993aaf45b55ea15ad9138a0e2b196e1813"

df = pd.read_csv(DATA_URL)
full_history_df = df.copy(deep=True)

print("Data loaded from:", DATA_URL)
print("Rows:", len(df), "| Companies:", df["ticker"].nunique())
df.head()


## 2. Keep each company’s most recent fiscal period

We select the latest period **before** checking Altman inputs. A firm is never moved to an older year merely to obtain a score.

**Data treatment:** the validation snapshot has 248 firm-year rows. Keeping one latest row per ticker removes 198 older observations and retains all 50 firms. This creates a comparable one-row-per-firm exercise, but Yahoo may later restate historical values and does not provide filing-publication timestamps.

Duplicate ticker/date rows are not silently removed. The code stops if it finds one because the correct observation would require investigation.


In [ ]:
df["fiscal_period_end"] = pd.to_datetime(df["fiscal_period_end"], errors="coerce")
full_history_df["fiscal_period_end"] = pd.to_datetime(
    full_history_df["fiscal_period_end"], errors="coerce"
)

duplicate_keys = full_history_df.duplicated(
    ["ticker", "fiscal_period_end"], keep=False
)
if duplicate_keys.any():
    display(full_history_df.loc[duplicate_keys].sort_values(["ticker", "fiscal_period_end"]))
    raise ValueError("Duplicate firm-period records require investigation.")

df = (
    df.sort_values(["ticker", "fiscal_period_end"], ascending=[True, False])
      .drop_duplicates("ticker", keep="first")
      .copy()
)

assert len(df) == EXPECTED_UNIVERSE_COUNT
assert df["ticker"].nunique() == EXPECTED_UNIVERSE_COUNT

print("Companies retained:", len(df))
df[["ticker", "fiscal_period_end", "sector", "industry"]].head()


## 3. Restore and label the Chapter 4 inputs

As in Chapter 3, we merge the selected ticker/date keys back to the preserved full-history data. The merge keeps exactly the latest observation already chosen for each firm.


In [ ]:
chapter4_numeric_columns = [
    "working_capital", "current_assets", "current_liabilities",
    "retained_earnings", "ebit", "ordinary_shares_number",
    "fiscal_period_end_close", "total_liabilities",
    "total_liabilities_net_minority_interest", "total_revenue", "total_assets",
]
for column in chapter4_numeric_columns:
    full_history_df[column] = pd.to_numeric(full_history_df[column], errors="coerce")

latest_keys = df[["ticker", "fiscal_period_end"]].copy()
master_df = latest_keys.merge(
    full_history_df,
    on=["ticker", "fiscal_period_end"],
    how="left",
    validate="one_to_one",
).sort_values("ticker").reset_index(drop=True)

assert len(master_df) == EXPECTED_UNIVERSE_COUNT
assert master_df[["ticker", "fiscal_period_end"]].duplicated().sum() == 0

master_df[["ticker", "company_name", "sector", "fiscal_period_end"]].head()


### Variable mapping and accounting judgments

> **Variable mapping: Working capital.** The textbook uses working capital. Yahoo supplies `working_capital`, which we use first. If it is missing, we calculate `current_assets - current_liabilities` only when both components exist. In this snapshot that fallback affects **0 firms**. Missing components are not zero.

> **Variable mapping: Retained earnings.** The textbook and Yahoo both use retained earnings. We use `retained_earnings` directly and do not substitute total equity because equity includes contributed capital and other items.

> **Variable mapping: EBIT.** The textbook uses earnings before interest and taxes. We use Yahoo’s `ebit` field directly. We do not silently replace missing EBIT with operating income.

> **Variable mapping: Market value of equity.** We calculate fiscal-date market equity as `ordinary_shares_number × fiscal_period_end_close`. This affects **50 firms** with both inputs. We deliberately do not use Yahoo’s current market capitalization because it would mix today’s price with an earlier fiscal-year balance sheet. The limitation is that the annual statement share count may not equal the exact shares outstanding on the price date.

> **Variable mapping: Book value of total liabilities.** Yahoo’s exact `total_liabilities` field is unpopulated in this saved dataset. We therefore use `total_liabilities_net_minority_interest`, Yahoo’s closest available book-liability total, for **49 firms**. This can include minority interest and may be slightly broader than the textbook denominator. The remaining unavailable observation stays missing.

> **Variable mapping: Sales and total assets.** We use Yahoo `total_revenue` for sales and `total_assets` for the common denominator.

No missing Yahoo value is automatically replaced with zero.


### Exercise 1 — Create the clearly labelled Altman inputs

Prepare `Altman_Working_Capital`, `Altman_Retained_Earnings`, `Altman_EBIT`, `Altman_Market_Equity`, `Altman_Total_Liabilities`, `Altman_Sales`, and `Altman_Total_Assets`. Also preserve a source label for every derived or substituted value.


In [ ]:
# Write or paste your code here, then press Run


## 4. The Chapter 4 Altman model

The course formula is:

\[
Z = 0.012X_1 + 0.014X_2 + 0.033X_3 + 0.006X_4 + 0.999X_5
\]

- `X1` = working capital / total assets: short-term liquidity relative to the asset base.
- `X2` = retained earnings / total assets: cumulative profitability retained in the business.
- `X3` = EBIT / total assets: operating earnings generated by the asset base.
- `X4` = market value of equity / book value of total liabilities: the market equity cushion relative to obligations.
- `X5` = sales / total assets: asset turnover.

### Scaling convention used for this calculation

The supplied Chapter 4 notes show the coefficients above but do not fully state the numeric scaling of each input. Scaling changes the score and its cutoff comparison. To make the displayed formula calculation-ready, this notebook expresses **X1 through X4 in percentage points** and leaves **X5 as the decimal sales/assets ratio**. This is numerically equivalent to applying coefficients 1.2, 1.4, 3.3, and 0.6 to decimal X1–X4 ratios while keeping 0.999 on X5.

This convention is explicit so it can be confirmed or revised by the instructor. We do not silently substitute a different Altman model.


### Raw Altman Z categories used in this notebook

The Chapter 4 source identifies **2.675 as Altman’s original cutoff** and **1.81 as the later distress boundary**. We preserve both without adding an unsupported 2.99 cutoff:

- `Raw Altman Z < 1.81`: **Below 1.81 distress boundary**. This is the worrisome range used for the Stage 1 flag.
- `1.81 ≤ Raw Altman Z < 2.675`: **Between 1.81 and 2.675**.
- `Raw Altman Z ≥ 2.675`: **At or above 2.675 original cutoff**.

Lower raw scores indicate greater distress concern. These ranges apply only to the **raw Altman Z**, never to the later industry-adjusted value.


### Exercise 2 — Calculate the raw Altman Z

Create the five components, require complete inputs plus positive assets and liabilities, and leave invalid scores missing. Do not use `.fillna(0)`.


In [ ]:
# Write or paste your code here, then press Run


### Exercise 3 — Assign the raw-score category and Stage 1 flag

Apply the Chapter 4 boundaries to the raw score only. The first-stage flag is `True` only below the 1.81 distress boundary.


In [ ]:
# Write or paste your code here, then press Run


## 5. Stage 1: absolute distress

**Stage 1 — Absolute distress:** Raw Altman Z asks whether the firm’s financial condition is concerning in absolute terms. Firms below the course’s 1.81 distress boundary are flagged for further investigation.

The flag is a screen, not proof that failure will occur. Altman’s original evidence came from a small sample of public manufacturers, so cross-industry interpretation requires care.


## 6. Stage 2: relative distress within broad sectors

Chapter 3 used Yahoo’s broad `sector` field for peer groups. We use the same grouping here.

\[
Relative\ Altman\ Z = Firm\ Raw\ Altman\ Z - Sector\ Mean\ Raw\ Altman\ Z
\]

This is **demeaning**, not statistical standardization. We subtract the sector mean and do not divide by a standard deviation.

- Negative relative value: lower and worse than the sector mean.
- Value near zero: approximately typical of the sector.
- Positive relative value: higher and better than the sector mean.

The relative measure adds context. It does not replace or override the raw classification. The 1.81 and 2.675 cutoffs must never be applied to the relative measure.


### Exercise 4 — Calculate the sector mean and Relative Altman Z

Use `groupby("sector")["Raw_Altman_Z"].transform(...)`. Require at least three valid raw scores in the sector before using its mean.


In [ ]:
# Write or paste your code here, then press Run


### Absolute versus relative distress

**Absolute distress:** Raw Altman Z evaluates the firm using the original Altman framework and the course boundaries.

**Relative distress:** Relative Altman Z shows how the same raw score compares with firms operating in a similar broad-sector environment.

A firm can have a concerning raw score but look approximately typical or stronger than its sector. The absolute signal still remains. The comparison suggests that some weakness may be common among peers; it does not prove why.

If a firm has both a concerning raw score and a negative relative value, the second stage reinforces the concern because the firm looks weak in absolute terms and weak relative to peers.


### Exercise 5 — Build the final two-stage distress table

Keep only Stage 1 flagged firms. For a transparent short interpretation, use the sign of Relative Altman Z after rounding to two displayed decimals. A missing relative value means the group failed the minimum-three rule.


In [ ]:
# Write or paste your code here, then press Run


## 7. Data Treatment Summary

The current validation snapshot produces **43 valid raw scores from 50 firms**. **7 firms** remain unscored because a required input is unavailable. We do not replace any of those missing inputs with zero.

The code below reports what changed, why, how many firms were affected, and the limitation of each treatment. It also identifies groups that fail the minimum-three benchmark rule.


In [ ]:
treatment_summary = pd.DataFrame([
    {
        "Variable / treatment": "Most recent fiscal period",
        "Firms affected": len(master_df),
        "What and why": f"Kept one latest row per ticker; removed {len(full_history_df) - len(master_df)} older rows so each firm enters once.",
        "Limitation": "Yahoo history may contain later restatements and lacks filing-publication timestamps.",
    },
    {
        "Variable / treatment": "Working capital derivation",
        "Firms affected": int(wc_fallback.sum()),
        "What and why": "Used current assets minus current liabilities only when direct working capital was missing and both components existed.",
        "Limitation": "Component classifications can differ across issuers.",
    },
    {
        "Variable / treatment": "Book liabilities substitution",
        "Firms affected": int(liabilities_fallback.sum()),
        "What and why": "Used Total Liabilities Net Minority Interest because exact Total Liabilities was unavailable.",
        "Limitation": "The substitute can include minority interest and may exceed the exact textbook denominator.",
    },
    {
        "Variable / treatment": "Fiscal-date market equity derivation",
        "Firms affected": int(master_df["Altman_Market_Equity"].notna().sum()),
        "What and why": "Multiplied fiscal-period-end raw close by annual ordinary shares to align market and accounting dates.",
        "Limitation": "Statement shares may not equal the exact shares outstanding on the price date.",
    },
    {
        "Variable / treatment": "Missing required inputs",
        "Firms affected": int((~master_df["Altman_Eligible"]).sum()),
        "What and why": "Left raw Altman Z missing instead of treating absent accounting data as zero.",
        "Limitation": "Coverage falls below the 50-firm starting universe.",
    },
    {
        "Variable / treatment": "Sector minimum-three rule",
        "Firms affected": int((
            master_df["Raw_Altman_Z"].notna()
            & master_df["Sector_Valid_Altman_N"].lt(MIN_SECTOR_OBSERVATIONS)
        ).sum()),
        "What and why": "Withheld Relative Altman Z when a sector had fewer than 3 valid firms.",
        "Limitation": "A firm can lack peer context even though its raw result remains available.",
    },
    {
        "Variable / treatment": "NaN-to-zero replacements",
        "Firms affected": 0,
        "What and why": "No missing Yahoo value was replaced with zero.",
        "Limitation": "None; unavailable required inputs remain visible.",
    },
])

display(treatment_summary)
master_df.loc[~master_df["Altman_Eligible"], [
    "ticker", "company_name", "sector", "Altman_Missing_Reason"
]].sort_values("ticker")


## 8. Verification and export

These checks confirm the 50-firm latest-period structure, finite scores, correct relative-score direction, minimum group size, and raw-only use of the course cutoffs. The private repository also contains independent scalar checks for several firms.


### Final check

After completing the five exercises, confirm that your final table contains only Stage 1 flagged firms and that no raw Altman cutoff has been applied to `Relative_Altman_Z`.
